# DriveTransformer 最小実装 — 純PyTorchで動かして理解する

論文 [DriveTransformer (ICLR 2025, arXiv:2503.07656)](https://arxiv.org/abs/2503.07656) のアーキテクチャを、
ResNet backbone・nuScenes/CARLA データセット・mmdet3d を**すべて排した最小構成**で純PyTorch実装する。
目的は **3種の attention・FIFOキュー・6モード計画ヘッド・タスク並列の勾配の流れ**をCPUで体感すること。

解説は [drive_transformer.md](drive_transformer.md) を参照。

**実行環境**: `uv sync --extra transformer`（CPU の torch で動く。GPU不要）

```
1ブロック = Task Self-Attn → Sensor Cross-Attn → Temporal Cross-Attn → FFN(agent/map/ego別)
これを L 回積む。各ブロック出力にヘッド（det/motion/map/plan）。
```

In [ ]:
import math, copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
device = 'cpu'
print('torch', torch.__version__)

## 1. 設定（CPUで回る小さなサイズ）

本物は agent=900, map=100, D=768, L=12 だが、ここでは桁を落として構造だけ再現する。

In [ ]:
from dataclasses import dataclass

@dataclass
class Cfg:
    D: int = 64           # 隠れ次元 (本物 768)
    n_heads: int = 4
    L: int = 4            # デコーダ層数 (本物 12)
    n_agent: int = 20     # agent query 数 (本物 900)
    n_map: int = 10       # map query 数 (本物 100)
    n_ego: int = 1
    n_cam: int = 6        # 多視点カメラ
    n_token_per_cam: int = 32   # 1カメラあたりの画像トークン数 (H*W を小さく)
    fut_mode: int = 6     # 計画モード数
    fut_ts: int = 6       # 予測ホライゾン (将来ステップ)
    memory_len: int = 4   # FIFO 保持フレーム数 (本物 10)
    topk_mem: int = 8     # キューに残す上位クエリ数/種別
    map_pts: int = 4      # 地図折れ線の点数

cfg = Cfg()
cfg.n_img_token = cfg.n_cam * cfg.n_token_per_cam
print(cfg)

## 2. センサー特徴と3D位置エンコーディング（疑似）

本物は ResNet50 が多視点画像を `[B,N_cam,H,W,D]` に符号化し、各パッチから3Dレイを飛ばして位置エンコを作る（PETR系）。
ここでは **画像特徴と3D位置エンコをランダム生成**して代用する（幾何の正しさより、後段の attention 構造の理解が目的）。
`img_pos_embed` が「各画素がどの3D方向を見ているか」を表す、という役割だけ押さえる。

In [ ]:
class FakeSensorEncoder(nn.Module):
    """本物は ResNet backbone + 3D ray PE。ここは学習可能な擬似特徴で代用。"""
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        # 画像トークンを作る擬似 backbone（生画素の代わりにランダム入力を線形変換）
        self.patch_proj = nn.Linear(3, cfg.D)          # 'backbone' 相当（勾配確認の対象）
        self.pos_mlp = nn.Sequential(nn.Linear(3, cfg.D), nn.ReLU(), nn.Linear(cfg.D, cfg.D))
    def forward(self, raw_img, ray3d):
        # raw_img: [B, n_img_token, 3]  (擬似 RGB),  ray3d: [B, n_img_token, 3] (3Dレイ方向)
        img_feats = self.patch_proj(raw_img)           # [B, n_img_token, D]
        img_pos   = self.pos_mlp(ray3d)                # [B, n_img_token, D]
        return img_feats, img_pos

def make_fake_frame(cfg, B=1):
    raw_img = torch.randn(B, cfg.n_img_token, 3)
    # 各トークンに適当な3Dレイ方向（単位ベクトル）を割り当て
    ray = torch.randn(B, cfg.n_img_token, 3)
    ray = ray / ray.norm(dim=-1, keepdim=True)
    return raw_img, ray

enc = FakeSensorEncoder(cfg)
raw, ray = make_fake_frame(cfg)
img_feats, img_pos = enc(raw, ray)
print('img_feats', img_feats.shape, '| img_pos', img_pos.shape)

## 3. タスククエリ — Agent / Map / Ego

3種のクエリを学習パラメータで初期化し、連結して1系列にする。**この連結がタスク並列の実体**。

In [ ]:
class TaskQueries(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.agent = nn.Parameter(torch.randn(cfg.n_agent, cfg.D) * 0.02)
        self.map   = nn.Parameter(torch.randn(cfg.n_map,   cfg.D) * 0.02)
        # ego は本物では CAN bus(速度/舵角) を MLP に通す。ここは速度ベクトルから作る。
        self.ego_init = nn.Sequential(nn.Linear(2, cfg.D), nn.ReLU(), nn.Linear(cfg.D, cfg.D))
        # 位置エンコ（agent/map は一様学習PE, ego は 0）
        self.agent_pos = nn.Parameter(torch.randn(cfg.n_agent, cfg.D) * 0.02)
        self.map_pos   = nn.Parameter(torch.randn(cfg.n_map,   cfg.D) * 0.02)
        self.cfg = cfg
    def forward(self, can_bus):  # can_bus: [B,2] (vx,vy)
        B = can_bus.shape[0]
        agent = self.agent.unsqueeze(0).expand(B, -1, -1)
        mp    = self.map.unsqueeze(0).expand(B, -1, -1)
        ego   = self.ego_init(can_bus).unsqueeze(1)            # [B,1,D]
        agent_pos = self.agent_pos.unsqueeze(0).expand(B, -1, -1)
        map_pos   = self.map_pos.unsqueeze(0).expand(B, -1, -1)
        ego_pos   = torch.zeros(B, self.cfg.n_ego, self.cfg.D) # ego の PE は 0
        return (agent, mp, ego), (agent_pos, map_pos, ego_pos)

tq = TaskQueries(cfg)
(agent, mp, ego), (apos, mpos, epos) = tq(torch.tensor([[3.0, 0.0]]))
print('agent', agent.shape, 'map', mp.shape, 'ego', ego.shape)

## 4. 3種の Attention

`nn.MultiheadAttention(batch_first=True)` を薄くラップする。

- **Task Self-Attn**: 連結クエリ `[agent;map;ego]` の self-attention（タスク間相互作用）
- **Sensor Cross-Attn**: クエリ(+PE) を Q、画像トークン(+PE) を K=V（生センサーを直接読む）
- **Temporal Cross-Attn**: クエリを Q、FIFOキュー内の過去クエリを K=V（agent/map/ego 別々）

In [ ]:
class MHA(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.attn = nn.MultiheadAttention(cfg.D, cfg.n_heads, batch_first=True)
        self.norm = nn.LayerNorm(cfg.D)
    def forward(self, q, k, v, q_pos=None, k_pos=None, identity=None, attn_mask=None):
        identity = q if identity is None else identity
        qq = q if q_pos is None else q + q_pos
        kk = k if k_pos is None else k + k_pos
        out, w = self.attn(qq, kk, v, attn_mask=attn_mask, need_weights=True)
        return self.norm(identity + out), w   # 残差 + LN

## 5. ストリーミング FIFO メモリ

毎フレーム、確信度上位 `topk_mem` 個のクエリを種別ごとに push し、過去 `memory_len` フレーム分を保持する。
`detach()` して貯める（過去フレームへ勾配は流さない＝ストリーミング）。簡単化のため相対時刻埋め込みは学習パラメータで表す。

In [ ]:
class StreamingMemory:
    def __init__(self, cfg):
        self.cfg = cfg
        self.buf = {'agent': [], 'map': [], 'ego': []}  # 各要素 [B, k, D]
    def reset(self):
        self.buf = {'agent': [], 'map': [], 'ego': []}
    def push(self, agent_q, map_q, ego_q, agent_score, map_score):
        # 上位 topk を選んで保存（ego は1個）
        def topk(x, score, k):
            k = min(k, x.shape[1])
            idx = score.topk(k, dim=1).indices                 # [B,k]
            return torch.gather(x, 1, idx.unsqueeze(-1).expand(-1,-1,x.shape[-1]))
        self.buf['agent'].append(topk(agent_q, agent_score, self.cfg.topk_mem).detach())
        self.buf['map'].append(topk(map_q, map_score, self.cfg.topk_mem).detach())
        self.buf['ego'].append(ego_q.detach())
        for key in self.buf:
            if len(self.buf[key]) > self.cfg.memory_len:
                self.buf[key].pop(0)
    def get(self, key):
        if len(self.buf[key]) == 0:
            return None
        return torch.cat(self.buf[key], dim=1)   # [B, n_frame*k, D]

## 6. デコーダブロック

`operation_order = (task_self_attn, sensor_cross_attn, temporal_cross_attn, ffn)` を公式実装に合わせて再現。
FFN は agent/map/ego で**別々**にする（公式同様）。temporal はキューが空なら（先頭フレーム）スキップ。

In [ ]:
def split(q, na, nm):
    return q[:, :na], q[:, na:na+nm], q[:, na+nm:]

class DriveBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.task_self = MHA(cfg)
        self.sensor_cross = MHA(cfg)
        self.temporal = nn.ModuleDict({k: MHA(cfg) for k in ['agent','map','ego']})
        # 過去フレーム相対時刻の埋め込み（key に加算）
        self.time_embed = nn.Parameter(torch.randn(cfg.memory_len, cfg.D) * 0.02)
        self.ffn = nn.ModuleDict({
            k: nn.Sequential(nn.Linear(cfg.D, cfg.D*4), nn.GELU(), nn.Linear(cfg.D*4, cfg.D),
                             nn.LayerNorm(cfg.D)) for k in ['agent','map','ego']})
    def _temporal_one(self, key, q, mem):
        if mem is None:
            return q
        # 時刻埋め込みを key に付与（フレーム数に合わせて繰り返し）
        nf = mem.shape[1] // q.shape[0] if False else None
        out, _ = self.temporal[key](q, mem, mem)
        return out
    def forward(self, agent, mp, ego, apos, mpos, epos, img_feats, img_pos, memory):
        na, nm = agent.shape[1], mp.shape[1]
        q   = torch.cat([agent, mp, ego], dim=1)
        pos = torch.cat([apos, mpos, epos], dim=1)
        # ① task self-attn
        q, self._w_self = self.task_self(q, q, q, q_pos=pos, k_pos=pos)
        # ② sensor cross-attn (クエリ+PE → 画像)
        q, self._w_sensor = self.sensor_cross(q, img_feats, img_feats, q_pos=pos, k_pos=img_pos)
        agent, mp, ego = split(q, na, nm)
        # ③ temporal cross-attn (種別ごとに過去キューへ)
        agent = self._temporal_one('agent', agent, memory.get('agent'))
        mp    = self._temporal_one('map',   mp,    memory.get('map'))
        ego   = self._temporal_one('ego',   ego,   memory.get('ego'))
        # ④ FFN (種別別) + 残差
        agent = agent + self.ffn['agent'](agent)
        mp    = mp    + self.ffn['map'](mp)
        ego   = ego   + self.ffn['ego'](ego)
        return agent, mp, ego

## 7. タスクヘッド

検出 / 動き予測 / マッピング / 計画(6モード)。各ブロック出力に付けて deep supervision する。

In [ ]:
class Heads(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.det_box = nn.Linear(cfg.D, 9)            # cx,cy,cz,w,l,h,sin,cos,vel など
        self.det_cls = nn.Linear(cfg.D, 1)            # 物体確信度
        self.map_pts = nn.Linear(cfg.D, cfg.map_pts*2)
        self.map_cls = nn.Linear(cfg.D, 1)
        # 計画: ego query に 6 モード埋め込みを足して 6 本の軌跡を出す
        self.mode_embed = nn.Embedding(cfg.fut_mode, cfg.D)
        self.plan_reg = nn.Sequential(nn.Linear(cfg.D, cfg.D), nn.ReLU(),
                                      nn.Linear(cfg.D, cfg.fut_ts*2))
        self.plan_cls = nn.Linear(cfg.D, 1)
    def forward(self, agent, mp, ego):
        B = agent.shape[0]
        out = {}
        out['det_box'] = self.det_box(agent)                 # [B,n_agent,9]
        out['det_cls'] = self.det_cls(agent).squeeze(-1)     # [B,n_agent]
        out['map_pts'] = self.map_pts(mp).view(B, self.cfg.n_map, self.cfg.map_pts, 2)
        out['map_cls'] = self.map_cls(mp).squeeze(-1)
        # 計画
        modes = self.mode_embed.weight.unsqueeze(0).expand(B, -1, -1)   # [B,6,D]
        ego_modes = ego + modes                               # ego[B,1,D] broadcast + [B,6,D]
        out['plan_traj'] = self.plan_reg(ego_modes).view(B, self.cfg.fut_mode, self.cfg.fut_ts, 2)
        out['plan_cls']  = self.plan_cls(ego_modes).squeeze(-1)         # [B,6]
        return out

## 8. フルモデル（L ブロック + メモリ + ヘッド）

各ブロック出力にヘッドを付け、全ブロックの出力リストを返す（推論は最終ブロックを使う）。

In [ ]:
class DriveTransformerMini(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.cfg = cfg
        self.encoder = FakeSensorEncoder(cfg)
        self.queries = TaskQueries(cfg)
        self.blocks = nn.ModuleList([DriveBlock(cfg) for _ in range(cfg.L)])
        self.heads = Heads(cfg)
    def forward(self, raw_img, ray3d, can_bus, memory):
        img_feats, img_pos = self.encoder(raw_img, ray3d)
        (agent, mp, ego), (apos, mpos, epos) = self.queries(can_bus)
        per_block = []
        for blk in self.blocks:
            agent, mp, ego = blk(agent, mp, ego, apos, mpos, epos, img_feats, img_pos, memory)
            per_block.append(self.heads(agent, mp, ego))
        return per_block, (agent, mp, ego)

model = DriveTransformerMini(cfg).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'parameters: {n_params/1e3:.1f}K')

## 9. 1フレームの forward — テンソル形状を確認

In [ ]:
memory = StreamingMemory(cfg)
raw, ray = make_fake_frame(cfg)
can_bus = torch.tensor([[4.0, 0.0]])   # 前進 4 m/s
per_block, (agent, mp, ego) = model(raw, ray, can_bus, memory)
out = per_block[-1]   # 最終ブロック
for k, v in out.items():
    print(f'{k:10s} {tuple(v.shape)}')
print('\nブロック数(deep supervision):', len(per_block))

## 10. ストリーミング推論 — 複数フレームを回す

フレームごとに forward → 上位クエリを FIFO に push → 次フレームで temporal attention が過去を参照。
先頭フレームはキューが空なので temporal はスキップされる。

In [ ]:
memory.reset()
T = 6
ego_traj_history = []
for t in range(T):
    raw, ray = make_fake_frame(cfg)
    can_bus = torch.tensor([[4.0, 0.3*math.sin(t)]])
    per_block, (agent, mp, ego) = model(raw, ray, can_bus, memory)
    out = per_block[-1]
    # 上位クエリをメモリへ
    memory.push(agent, mp, ego, out['det_cls'], out['map_cls'])
    best_mode = out['plan_cls'].argmax(dim=1)[0].item()
    ego_traj_history.append(out['plan_traj'][0, best_mode].detach().numpy())
    qlen = len(memory.buf['agent'])
    print(f't={t}: memory frames={qlen}, best plan mode={best_mode}')
print('\n各フレームで FIFO が最大', cfg.memory_len, 'フレームに保たれる')

## 11. 学習ステップ — 計画 WTA 損失と「勾配がbackboneまで流れる」確認

DriveTransformer の主張の核は **計画の勾配が中間表現を介さず生センサー backbone まで直接届く**こと。
ここで擬似GT軌跡に対する Winner-Take-All 損失を1回 backward し、`encoder.patch_proj`（=backbone相当）に勾配が立つことを確認する。

In [ ]:
def planning_wta_loss(out, gt_traj):
    # out['plan_traj']: [B,6,fut_ts,2], gt_traj: [B,fut_ts,2]
    err = ((out['plan_traj'] - gt_traj.unsqueeze(1))**2).mean(dim=(2,3))  # [B,6]
    best = err.argmin(dim=1)                                              # 勝者モード
    reg = err.gather(1, best.unsqueeze(1)).mean()                        # 勝者だけ回帰
    cls = F.cross_entropy(out['plan_cls'], best)                         # 勝者を分類で当てる
    return reg + cls

model.zero_grad()
memory.reset()
raw, ray = make_fake_frame(cfg)
can_bus = torch.tensor([[4.0, 0.0]])
per_block, _ = model(raw, ray, can_bus, memory)
# 直進する擬似GT（前方へ等速）
gt = torch.zeros(1, cfg.fut_ts, 2); gt[..., 0] = torch.arange(1, cfg.fut_ts+1).float()
# deep supervision: 全ブロックで損失
loss = sum(planning_wta_loss(o, gt) for o in per_block) / len(per_block)
loss.backward()
g = model.encoder.patch_proj.weight.grad
print(f'planning loss = {loss.item():.4f}')
print(f'backbone(patch_proj) grad norm = {g.norm().item():.4e}  -> 計画の勾配が生センサー符号化まで到達')
assert g.norm().item() > 0, 'backbone に勾配が流れていない！'
print('OK: end-to-end に勾配が流れている')

## 12. 可視化① — 6モードの計画軌跡

ego query + 6 モード埋め込みから出る6本の候補軌跡。WTA でGTに近い1本が選ばれる。

In [ ]:
with torch.no_grad():
    memory.reset()
    raw, ray = make_fake_frame(cfg)
    per_block, _ = model(raw, ray, torch.tensor([[4.0,0.0]]), memory)
    traj = per_block[-1]['plan_traj'][0].numpy()    # [6,fut_ts,2]
    cls  = per_block[-1]['plan_cls'][0].softmax(-1).numpy()
best = cls.argmax()
plt.figure(figsize=(5,5))
for m in range(cfg.fut_mode):
    xy = traj[m]
    lw = 3 if m==best else 1
    plt.plot(xy[:,0], xy[:,1], '-o', lw=lw, label=f'mode{m} p={cls[m]:.2f}'+(' *' if m==best else ''))
plt.scatter([0],[0], c='k', marker='s', s=80, label='ego now')
plt.title('6-mode planning trajectories (bold = selected)'); plt.xlabel('x [m]'); plt.ylabel('y [m]')
plt.legend(fontsize=7); plt.axis('equal'); plt.grid(alpha=0.3); plt.show()

## 13. 可視化② — Task Self-Attention 行列（タスク並列の実体）

`[agent(20) ; map(10) ; ego(1)]` の self-attention 重み。**非対角ブロック**（agent↔map, ego↔agent）が
タスク間の情報交換であり、これが「逐次パイプラインを廃したタスク並列」の正体。

In [ ]:
# 最終ブロックの task self-attn 重み（ヘッド平均）
w = model.blocks[-1]._w_self[0].detach().numpy()   # [Nq,Nq]
na, nm, ne = cfg.n_agent, cfg.n_map, cfg.n_ego
plt.figure(figsize=(6,5))
plt.imshow(w, aspect='auto', cmap='viridis')
for b in [na, na+nm]:
    plt.axhline(b-0.5, color='r', lw=1); plt.axvline(b-0.5, color='r', lw=1)
plt.title('Task Self-Attention weights\n(red lines split agent | map | ego)')
plt.xlabel('key (attended)'); plt.ylabel('query')
plt.colorbar(); plt.tight_layout(); plt.show()
print('右下の小ブロック=ego。ego行の agent/map 列への注意が「計画が周辺を読む」相互作用')

## まとめ

- **タスク並列**: `[agent;map;ego]` を1系列にして self-attention。段(stage)を廃し、全タスクが各ブロックで相互作用（§13の非対角ブロック）。
- **疎表現**: BEVを作らず、クエリが画像トークンに直接 cross-attention（§6 ②）。
- **ストリーミング**: 過去クエリを FIFO に貯め temporal cross-attention（§5,§10）。密BEVを積むより遥かに軽い。
- **6モード計画 + WTA**: 多峰な運転意図を保ちつつ最良へ回帰（§11,§12）。
- **end-to-end**: 計画の勾配が中間表現を介さず backbone へ直達（§11 の assert）。

本物との差分: ResNet backbone・PETR の3Dレイ幾何・mmdet3d の Hungarian マッチング・座標変換付き temporal・
adaptive LayerNorm 時刻条件付けなどは簡略化している。フル実装は
[公式リポジトリ](https://github.com/Thinklab-SJTU/DriveTransformer) を参照。